In [2]:
from datasets import load_dataset
import pandas as pd
 
# Ladda IMDb-datasetet
dataset = load_dataset("stanfordnlp/imdb")

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
df_train_full = pd.DataFrame(dataset["train"])
df_test_full = pd.DataFrame(dataset["test"])

pd.set_option('display.max_colwidth', None)

df_train_full.head()

,text,label
0,"I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered ""controversial"" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot like some cheaply made porno. While my countrymen mind find it shocking, in reality sex and nudity are a major staple in Swedish cinema. Even Ingmar Bergman, arguably their answer to good old boy John Ford, had sex scenes in his films.<br /><br />I do commend the filmmakers for the fact that any sex shown in the film is shown for artistic purposes rather than just to shock people and make money to be shown in pornographic theaters in America. I AM CURIOUS-YELLOW is a good film for anyone wanting to study the meat and potatoes (no pun intended) of Swedish cinema. But really, this film doesn't have much of a plot.",0
1,"""I Am Curious: Yellow"" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don't exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies like The Brown Bunny, in which we're treated to the site of Vincent Gallo's throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) ""double-standard"" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious anatomical difference between men and women: there are no genitals on display when actresses appears nude, and the same cannot be said for a man. In fact, you generally won't see female genitals in an American film in anything short of porn or explicit erotica. This alleged double-standard is less a double standard than an admittedly depressing ability to come to terms culturally with the insides of women's bodies.",0
2,"If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />",0
3,"This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been r

In [5]:
print(df_train_full.info())

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    25000 non-null  str  
 1   label   25000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 32.0 MB
None


In [6]:
common_text = set(df_train_full["text"]) & set(df_test_full["text"])
print("Recensioner som finns i både train och test:", len(common_text))

Recensioner som finns i både train och test: 123


In [7]:
df_test = df_test_full[
    ~df_test_full["text"].isin(common_text)
].copy()
 
print("Test före:", len(df_test_full))
print("Test efter:", len(df_test))


Test före: 25000
Test efter: 24877


In [8]:
common_text = set(df_train_full["text"]) & set(df_test["text"])
print("Recensioner som finns i både train och test:", len(common_text))

Recensioner som finns i både train och test: 0


In [9]:
from sklearn.model_selection import train_test_split

X = df_train_full["text"]
y = df_train_full["label"]

X_test = df_test["text"]
y_test = df_test["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
import re

def clean_text(text): 
    # tar bort HTML-taggar
    text = re.sub(r'<.*?>', ' ', text)
    # gör om till små bokstäver
    text = text.lower()

    return text

In [11]:
X_train_clean = X_train.apply(clean_text)
X_val_clean = X_val.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
 
tfidf = TfidfVectorizer()

In [13]:
X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_val_tfidf = tfidf.transform(X_val_clean)
X_test_tfidf = tfidf.transform(X_test_clean)

In [14]:
from sklearn.linear_model import LogisticRegression
 
model = LogisticRegression(max_iter=1000, random_state=42)
 
model.fit(X_train_tfidf, y_train)

y_val_pred = model.predict(X_val_tfidf)

In [15]:
from sklearn.metrics import accuracy_score, classification_report
 
accuracy = accuracy_score(y_val, y_val_pred)
 
print("Accuracy:", accuracy)
print()
print(classification_report(y_val, y_val_pred))
 

Accuracy: 0.8886

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      2500
           1       0.88      0.90      0.89      2500

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.89      0.89      0.89      5000



DistilBERT

In [ ]:
train_texts = set(dataset["train"]["text"])
test_texts = set(dataset["test"]["text"])

common_texts = train_texts & test_texts

print("Antal recensioner som finns i både train och test:", len(common_texts))

In [ ]:
test_dataset_clean = dataset["test"].filter(
    lambda example: example["text"] not in common_texts
)

print("Original test:", len(dataset["test"]))
print("Rent test:", len(test_dataset_clean))

In [ ]:
train_test_split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)

In [ ]:
print(train_test_split)

In [ ]:
train_dataset = train_test_split["train"]
val_dataset = train_test_split["test"]

In [ ]:
print(train_dataset)
print(val_dataset)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

print(train_tokenized)

In [ ]:
train_tokenized.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_tokenized.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
from transformers import AutoModelForSequenceClassification

bert_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model = bert_model.to(device)

print(device)

In [ ]:
small_train_dataset = train_tokenized.shuffle(seed=42).select(range(5000))

print("Antal träningsrecensioner:", len(small_train_dataset))

In [ ]:
from torch.utils.data import DataLoader

batch_size = 16

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_tokenized,
    batch_size=batch_size
)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    bert_model.parameters(),
    lr=5e-5
)

In [ ]:
from tqdm.auto import tqdm

bert_model.train()

for batch in tqdm(small_train_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    optimizer.zero_grad()

    outputs = bert_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss

    loss.backward()
    optimizer.step()

print("Träningen är klar!")

Validation

In [ ]:
bert_model.eval()

val_predictions = []
val_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        val_predictions.extend(predictions.cpu().tolist())
        val_labels.extend(labels.cpu().tolist())

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    val_labels,
    val_predictions
)

print("Validation accuracy:", accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    val_labels,
    val_predictions
))